# FreshCheck C+Patch A100 Runner

Notebook ??????????????????????? `C+Patch` ?? Colab ??????? `A100 GPU` ??????? seed

??????? notebook ?????:
- clone/update repo ??? GitHub
- mount Google Drive
- ??????? dependencies
- ??? `C+Patch` ??? patch sampling ?????? `efficientnet_b0`, `swin_t`, `dinov2_vits14`
- evaluate ?? target holdout ??? source holdout
- ?????????? seed ???? mean/std ?????????

Runtime ???????????:
- `Python 3`
- `A100 GPU`
- `High-RAM: ON`
- Runtime version: `?????? (?????)`

???????? A100 ?????? `L4 GPU` ?????????????????


In [ ]:
#@title 1) Clone / Update repository
REPO_URL = "https://github.com/techasit239/DADS7202_PigPicture.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/DADS7202_PigPicture"  #@param {type:"string"}

import subprocess
from pathlib import Path

repo_path = Path(REPO_DIR)
if repo_path.exists():
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_path)], check=True)

%cd /content/DADS7202_PigPicture


In [ ]:
#@title 2) Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/FreshCheck')
ARTIFACTS_ROOT = DRIVE_ROOT / 'artifacts'
EXPERIMENTS_ROOT = ARTIFACTS_ROOT / 'experiments_manual'
EXPERIMENTS_AUTO_ROOT = ARTIFACTS_ROOT / 'experiments_auto'
TARGET_SPLIT_ROOT = ARTIFACTS_ROOT / 'target_split'

print('Drive root:', DRIVE_ROOT)
print('Artifacts :', ARTIFACTS_ROOT)


In [ ]:
#@title 3) Install dependencies
%cd /content/DADS7202_PigPicture
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt timm transformers


In [ ]:
#@title 3.1) Verify CLI supports C+Patch arguments
%cd /content/DADS7202_PigPicture
!python run_freshcheck.py train --help | head -n 120

print('
Expected flags should appear above:')
print('--patch-sampling')
print('--num-patches')
print('--freeze-backbone-epochs')
print('--amp')
print('--prefetch-factor')


## ??? cell 3.1 ??????? flags ????

?????? Colab clone repo ??????????????? GitHub ??????????????????

?????????????????????????????????:
1. push ?????????????? GitHub ??????? cell 1 ????
2. ??????????????????????????????? Colab/repo ???????

??????? cell 8-10 ??? ??? cell 3.1 ?????????? flags ????????


In [ ]:
#@title 4) Runtime check
import torch

print('cuda available :', torch.cuda.is_available())
print('device count    :', torch.cuda.device_count())
if torch.cuda.is_available():
    print('device name     :', torch.cuda.get_device_name(0))
    print('capability      :', torch.cuda.get_device_capability(0))

assert torch.cuda.is_available(), 'GPU not found. Please switch runtime to A100 GPU.'


In [ ]:
#@title 5) Experiment paths
TRAIN_CSV = str(EXPERIMENTS_ROOT / 'exp_c_train.csv')
VAL_CSV = str(EXPERIMENTS_ROOT / 'exp_c_val.csv')
TARGET_CSV = str(TARGET_SPLIT_ROOT / 'target_holdout.csv')
SOURCE_CSV = str(EXPERIMENTS_AUTO_ROOT / 'source_holdout.csv')
PATCH_ROOT = str(ARTIFACTS_ROOT / 'exp_cplus_patch_runs')

for p in [TRAIN_CSV, VAL_CSV, TARGET_CSV, SOURCE_CSV]:
    print(p)


In [ ]:
#@title 6) Choose global settings
SEEDS = "42 52 62 72 82"  #@param {type:"string"}
IMG_SIZE = 224  #@param {type:"integer"}
NUM_PATCHES = 2  #@param {type:"integer"}
NUM_WORKERS = 4  #@param {type:"integer"}
PREFETCH_FACTOR = 4  #@param {type:"integer"}
AMP_DTYPE = "bf16"  #@param ["bf16", "fp16"]
RUN_SOURCE_EVAL = True  #@param {type:"boolean"}

SEED_LIST = [int(x) for x in SEEDS.split()]
print('Seeds:', SEED_LIST)
print('Patch root:', PATCH_ROOT)


In [ ]:
#@title 7) Helper to run shell commands
import subprocess

def run_cmd(cmd: str):
    print(cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')


In [ ]:
#@title 8) Train + eval EfficientNet-B0
for seed in SEED_LIST:
    out = f"{PATCH_ROOT}/efficientnet_b0/seed_{seed}"
    train_cmd = f'''python run_freshcheck.py train \
  --train-csv "{TRAIN_CSV}" \
  --val-csv "{VAL_CSV}" \
  --output-dir "{out}/train" \
  --models efficientnet_b0 \
  --epochs 14 \
  --patience 4 \
  --lr 0.0003 \
  --weight-decay 0.005 \
  --dropout 0.25 \
  --img-size {IMG_SIZE} \
  --batch-size 32 \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --patch-sampling \
  --num-patches {NUM_PATCHES} \
  --freeze-backbone-epochs 2 \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
    run_cmd(train_cmd)

    target_cmd = f'''python run_freshcheck.py evaluate \
  --csv "{TARGET_CSV}" \
  --output-dir "{out}/eval_target" \
  --models efficientnet_b0 \
  --checkpoint-paths efficientnet_b0="{out}/train/checkpoints/efficientnet_b0_best.pt" \
  --img-size {IMG_SIZE} \
  --batch-size 32 \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --patch-sampling \
  --num-patches {NUM_PATCHES} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
    run_cmd(target_cmd)

    if RUN_SOURCE_EVAL:
        source_cmd = f'''python run_freshcheck.py evaluate \
  --csv "{SOURCE_CSV}" \
  --output-dir "{out}/eval_source" \
  --models efficientnet_b0 \
  --checkpoint-paths efficientnet_b0="{out}/train/checkpoints/efficientnet_b0_best.pt" \
  --img-size {IMG_SIZE} \
  --batch-size 32 \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --patch-sampling \
  --num-patches {NUM_PATCHES} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
        run_cmd(source_cmd)


In [ ]:
#@title 9) Train + eval Swin-T
for seed in SEED_LIST:
    out = f"{PATCH_ROOT}/swin_t/seed_{seed}"
    train_cmd = f'''python run_freshcheck.py train \
  --train-csv "{TRAIN_CSV}" \
  --val-csv "{VAL_CSV}" \
  --output-dir "{out}/train" \
  --models swin_t \
  --epochs 12 \
  --patience 4 \
  --lr 0.0003 \
  --weight-decay 0.01 \
  --dropout 0.30 \
  --img-size {IMG_SIZE} \
  --batch-size 16 \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --patch-sampling \
  --num-patches {NUM_PATCHES} \
  --freeze-backbone-epochs 3 \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
    run_cmd(train_cmd)

    target_cmd = f'''python run_freshcheck.py evaluate \
  --csv "{TARGET_CSV}" \
  --output-dir "{out}/eval_target" \
  --models swin_t \
  --checkpoint-paths swin_t="{out}/train/checkpoints/swin_t_best.pt" \
  --img-size {IMG_SIZE} \
  --batch-size 16 \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --patch-sampling \
  --num-patches {NUM_PATCHES} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
    run_cmd(target_cmd)

    if RUN_SOURCE_EVAL:
        source_cmd = f'''python run_freshcheck.py evaluate \
  --csv "{SOURCE_CSV}" \
  --output-dir "{out}/eval_source" \
  --models swin_t \
  --checkpoint-paths swin_t="{out}/train/checkpoints/swin_t_best.pt" \
  --img-size {IMG_SIZE} \
  --batch-size 16 \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --patch-sampling \
  --num-patches {NUM_PATCHES} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
        run_cmd(source_cmd)


In [ ]:
#@title 10) Train + eval DINOv2-ViT-S/14
for seed in SEED_LIST:
    out = f"{PATCH_ROOT}/dinov2_vits14/seed_{seed}"
    train_cmd = f'''python run_freshcheck.py train \
  --train-csv "{TRAIN_CSV}" \
  --val-csv "{VAL_CSV}" \
  --output-dir "{out}/train" \
  --models dinov2_vits14 \
  --epochs 10 \
  --patience 3 \
  --lr 0.001 \
  --weight-decay 0.01 \
  --dropout 0.30 \
  --img-size {IMG_SIZE} \
  --batch-size 16 \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --patch-sampling \
  --num-patches {NUM_PATCHES} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
    run_cmd(train_cmd)

    target_cmd = f'''python run_freshcheck.py evaluate \
  --csv "{TARGET_CSV}" \
  --output-dir "{out}/eval_target" \
  --models dinov2_vits14 \
  --checkpoint-paths dinov2_vits14="{out}/train/checkpoints/dinov2_vits14_best.pt" \
  --img-size {IMG_SIZE} \
  --batch-size 16 \
  --num-workers {NUM_WORKERS} \
  --prefetch-factor {PREFETCH_FACTOR} \
  --patch-sampling \
  --num-patches {NUM_PATCHES} \
  --amp \
  --amp-dtype {AMP_DTYPE} \
  --allow-tf32 \
  --persistent-workers \
  --seed {seed}'''
    run_cmd(target_cmd)


In [ ]:
#@title 11) Summarize target-holdout results across seeds
SUMMARY_ROOT = f"{PATCH_ROOT}/_summary"

run_cmd(f'''python summarize_seed_runs.py \
  --root-dir "{PATCH_ROOT}/efficientnet_b0" \
  --model efficientnet_b0 \
  --eval-subdir eval_target \
  --output-dir "{SUMMARY_ROOT}/efficientnet_b0"''')

run_cmd(f'''python summarize_seed_runs.py \
  --root-dir "{PATCH_ROOT}/swin_t" \
  --model swin_t \
  --eval-subdir eval_target \
  --output-dir "{SUMMARY_ROOT}/swin_t"''')

run_cmd(f'''python summarize_seed_runs.py \
  --root-dir "{PATCH_ROOT}/dinov2_vits14" \
  --model dinov2_vits14 \
  --eval-subdir eval_target \
  --output-dir "{SUMMARY_ROOT}/dinov2_vits14"''')


In [ ]:
#@title 12) View mean/std tables
import json
import pandas as pd
from pathlib import Path

summary_root = Path(f"{PATCH_ROOT}/_summary")
rows = []
for model_name in ['efficientnet_b0', 'swin_t', 'dinov2_vits14']:
    path = summary_root / model_name / f'{model_name}_seed_summary.json'
    if not path.exists():
        print('missing:', path)
        continue
    payload = json.loads(path.read_text(encoding='utf-8'))
    row = {'model': model_name, 'n_runs': payload['n_runs']}
    for key, value in payload['mean'].items():
        row[f'{key}_mean'] = value
    for key, value in payload['std'].items():
        row[f'{key}_std'] = value
    rows.append(row)

df = pd.DataFrame(rows)
display(df)


In [ ]:
#@title 13) Optional: inspect one confusion matrix image per model
from IPython.display import Image, display
from pathlib import Path

for model_name in ['efficientnet_b0', 'swin_t', 'dinov2_vits14']:
    sample_seed = SEED_LIST[0]
    path = Path(f"{PATCH_ROOT}/{model_name}/seed_{sample_seed}/eval_target/{model_name}_confusion_matrix.png")
    print('
', model_name, path)
    if path.exists():
        display(Image(filename=str(path)))
